## a) N gram Language model 

In [172]:
# Define text
text = "a b A b b c a a C b c a B B c a c b"

In [173]:
# lower the text for consistency and reduce variance
lowered_text = text.lower()
lowered_text

'a b a b b c a a c b c a b b c a c b'

In [174]:
# tokenize the text (space)
tokens = lowered_text.split()
tokens

['a',
 'b',
 'a',
 'b',
 'b',
 'c',
 'a',
 'a',
 'c',
 'b',
 'c',
 'a',
 'b',
 'b',
 'c',
 'a',
 'c',
 'b']

In [175]:
# function to n-gram the tokenized text
def n_gram(tokens, n):
    nGrammed_list = []
    for i in range(len(tokens)-n+1):
        nGrammed_list.append(tuple([tokens[j+i] for j in range(n)]))
    return nGrammed_list

In [176]:
# one grammed tokens
one_grammed = n_gram(tokens=tokens, n=1)
one_grammed

[('a',),
 ('b',),
 ('a',),
 ('b',),
 ('b',),
 ('c',),
 ('a',),
 ('a',),
 ('c',),
 ('b',),
 ('c',),
 ('a',),
 ('b',),
 ('b',),
 ('c',),
 ('a',),
 ('c',),
 ('b',)]

In [177]:
# 2 grammed text
two_grammed = n_gram(tokens, n = 2)
two_grammed

[('a', 'b'),
 ('b', 'a'),
 ('a', 'b'),
 ('b', 'b'),
 ('b', 'c'),
 ('c', 'a'),
 ('a', 'a'),
 ('a', 'c'),
 ('c', 'b'),
 ('b', 'c'),
 ('c', 'a'),
 ('a', 'b'),
 ('b', 'b'),
 ('b', 'c'),
 ('c', 'a'),
 ('a', 'c'),
 ('c', 'b')]

In [178]:
# 3 grammed text
three_grammed  = n_gram(tokens, n=3)
three_grammed

[('a', 'b', 'a'),
 ('b', 'a', 'b'),
 ('a', 'b', 'b'),
 ('b', 'b', 'c'),
 ('b', 'c', 'a'),
 ('c', 'a', 'a'),
 ('a', 'a', 'c'),
 ('a', 'c', 'b'),
 ('c', 'b', 'c'),
 ('b', 'c', 'a'),
 ('c', 'a', 'b'),
 ('a', 'b', 'b'),
 ('b', 'b', 'c'),
 ('b', 'c', 'a'),
 ('c', 'a', 'c'),
 ('a', 'c', 'b')]

In [179]:
# Create a set of n-grammed tokens (later for freq count use)
n_gramed_tokens = n_gram(tokens, n = 2)
my_set = set(n_gramed_tokens)

In [180]:
# create the dictionary to count the frequency of n grammed tokens
freq_dic = {token:0 for token in my_set}

# count the freq
for n_gramed_token in n_gramed_tokens:
    freq_dic[n_gramed_token] += 1

# print the frequency distribution
for token in freq_dic.keys():
    print(f"{str(token):10s} -> {freq_dic[token]}")

('b', 'b') -> 2
('a', 'a') -> 1
('c', 'a') -> 3
('b', 'a') -> 1
('c', 'b') -> 2
('a', 'c') -> 2
('b', 'c') -> 3
('a', 'b') -> 3


## b) N-Gram Model with Laplace Smoothing

In [181]:
# input the text corpus and value of `n`
text = "a b a. a c a."
n = 2

In [182]:
# convert into lowercase
lowr = text.lower().strip()

# add <START> and <END> labels for each sentence
labelled = "<START> "
for i in range(len(lowr)):
    if lowr[i] == '\n':
        labelled += " EOL "
        
    elif lowr[i] == ".":
        if i == len(lowr) - 1:
            labelled += " <END>"
        else:
            if lowr[i+1].isspace():
                labelled += " <END> <START>"
    else:
        labelled += lowr[i]

# tokenize
tokens = labelled.split()

# n grammed
n_grams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

# n-1 grammed
n_minus_one_grams = [tuple(tokens[i:i+(n-1)]) for i in range(len(tokens) - (n-1) + 1)]

lowr, labelled, tokens, n_grams, n_minus_one_grams

('a b a. a c a.',
 '<START> a b a <END> <START> a c a <END>',
 ['<START>', 'a', 'b', 'a', '<END>', '<START>', 'a', 'c', 'a', '<END>'],
 [('<START>', 'a'),
  ('a', 'b'),
  ('b', 'a'),
  ('a', '<END>'),
  ('<END>', '<START>'),
  ('<START>', 'a'),
  ('a', 'c'),
  ('c', 'a'),
  ('a', '<END>')],
 [('<START>',),
  ('a',),
  ('b',),
  ('a',),
  ('<END>',),
  ('<START>',),
  ('a',),
  ('c',),
  ('a',),
  ('<END>',)])

In [183]:
# vocab and vocab_size
vocab_set = set(tokens)
vocab_len = len(vocab_set)
vocab_len, vocab_set

(5, {'<END>', '<START>', 'a', 'b', 'c'})

In [184]:
# set of n-1 grams (or context)
n_minus_one_grams_set = set(n_minus_one_grams)

# dictinary that stores each context and occurence
context_freq_dic = {n_1_gram:0 for n_1_gram in n_minus_one_grams_set}

# populate the dict
for n_1_gram in n_minus_one_grams:
    print(n_1_gram)
    context_freq_dic[n_1_gram] += 1
context_freq_dic

('<START>',)
('a',)
('b',)
('a',)
('<END>',)
('<START>',)
('a',)
('c',)
('a',)
('<END>',)


{('c',): 1, ('b',): 1, ('<END>',): 2, ('a',): 4, ('<START>',): 2}

In [185]:
# each context as dict key and each word with its occurence count as value
freq_count_dict = {n_minus_one_gram: {word:0 for word in vocab_set} for n_minus_one_gram in n_minus_one_grams_set}

# populate the dic with freq counts
for ngram in n_grams:
    # extract the context
    context = ngram[:-1]
    # extract the word
    word = ngram[-1]
    # update the occurence count
    freq_count_dict[context][word]+=1

#  Text: '<START> a b a <END> <START> a c a <END>'
freq_count_dict

{('c',): {'a': 1, '<END>': 0, '<START>': 0, 'c': 0, 'b': 0},
 ('b',): {'a': 1, '<END>': 0, '<START>': 0, 'c': 0, 'b': 0},
 ('<END>',): {'a': 0, '<END>': 0, '<START>': 1, 'c': 0, 'b': 0},
 ('a',): {'a': 0, '<END>': 2, '<START>': 0, 'c': 1, 'b': 1},
 ('<START>',): {'a': 2, '<END>': 0, '<START>': 0, 'c': 0, 'b': 0}}

In [186]:
probability_model_dict = {}

# apply laplace smoothing
for context in freq_count_dict.keys():
    probability_model_dict[context] = {}
    
    for word in freq_count_dict[context].keys():
       probability_model_dict[context][word] = (freq_count_dict[context][word]+1) / (context_freq_dic[context] + vocab_len)
        
# printing smoothed probability
probability_model_dict

{('c',): {'a': 0.3333333333333333,
  '<END>': 0.16666666666666666,
  '<START>': 0.16666666666666666,
  'c': 0.16666666666666666,
  'b': 0.16666666666666666},
 ('b',): {'a': 0.3333333333333333,
  '<END>': 0.16666666666666666,
  '<START>': 0.16666666666666666,
  'c': 0.16666666666666666,
  'b': 0.16666666666666666},
 ('<END>',): {'a': 0.14285714285714285,
  '<END>': 0.14285714285714285,
  '<START>': 0.2857142857142857,
  'c': 0.14285714285714285,
  'b': 0.14285714285714285},
 ('a',): {'a': 0.1111111111111111,
  '<END>': 0.3333333333333333,
  '<START>': 0.1111111111111111,
  'c': 0.2222222222222222,
  'b': 0.2222222222222222},
 ('<START>',): {'a': 0.42857142857142855,
  '<END>': 0.14285714285714285,
  '<START>': 0.14285714285714285,
  'c': 0.14285714285714285,
  'b': 0.14285714285714285}}

## Exercise
1) Design a language model for auto completion of sentence using N-
Gram

In [188]:
# since we have already build the probability model dictioanry, we will be using it for the predictions

# return top next K words leared from the corpus (probability model dictioanry)
def predict_next_k_word(text, k):
    # lower
    text = text.lower()
    # tokenize
    tokens = text.split()
    # grab last n tokens
    last_n_minus_one_gram = tuple(tokens[-n:])
    # get probabilty
    word_probablity_distrib = probability_model_dict[last_n_minus_one_gram]

    # return k-words with maximum probabilty
    sorted_dic = dict(sorted(word_probablity_distrib.items(), key=lambda item: item[1], reverse=True))
    return list(sorted_dic)[:k]

In [189]:
text = 'a'
predictions = predict_next_k_word(text, 2)

print(f"Text: {text}")
print(f"Next possible words:{set(predictions)}")

Text: a
Next possible words:{'<END>', 'c'}
